# Avro Basics — 08: Avro vs Parquet


Detailed comparison of Avro and Parquet across all dimensions
to help you decide which to use for each part of your pipeline.

Topics: storage size, write/read speed, column pruning, predicate pushdown,
schema evolution, use case mapping.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:06:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Avro JAR: spark-avro_2.13-4.0.2


## Step 1 — Setup: Same Dataset, Both Formats



In [2]:

import random, datetime, pathlib
random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA"]

df_bench = spark.range(N).select(
    F.col("id").alias("order_id"),
    (F.col("id") % 50000 + 1).alias("customer_id"),
    F.element_at(F.array([F.lit(c) for c in CATEGORIES]),(F.col("id")%5+1).cast("int")).alias("category"),
    F.element_at(F.array([F.lit(c) for c in COUNTRIES]), (F.col("id")%6+1).cast("int")).alias("country"),
    F.round(F.rand(42)*500+10,2).alias("revenue"),
    F.rand(43).alias("score"),
    (F.date_add(F.lit("2024-01-01"),(F.col("id")%365).cast("int"))).cast("date").alias("order_date"),
)

AVRO_BENCH = f"{DATA_DIR}/bench_avro"
PQ_BENCH   = f"{DATA_DIR}/bench_parquet"

df_bench.write.format("avro").mode("overwrite").save(AVRO_BENCH)
df_bench.write.mode("overwrite").parquet(PQ_BENCH)

import glob as glib
avro_mb = sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{AVRO_BENCH}/*.avro"))/1024/1024
pq_mb   = sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{PQ_BENCH}/*.parquet"))/1024/1024
print(f"Same data: Avro={avro_mb:.1f} MB  Parquet={pq_mb:.1f} MB  (Parquet {avro_mb/pq_mb:.1f}x smaller)")


Same data: Avro=5.0 MB  Parquet=4.1 MB  (Parquet 1.2x smaller)


## Step 2 — Read Performance Benchmark



In [3]:

benchmarks = [
    ("Full scan + sum",
     lambda p,f: spark.read.format(f).load(p).agg(F.sum("revenue")).collect()),
    ("Filter + count",
     lambda p,f: spark.read.format(f).load(p).filter(F.col("category")=="Electronics").count()),
    ("Select 2 cols",
     lambda p,f: spark.read.format(f).load(p).select("revenue","category").agg(F.sum("revenue")).collect()),
    ("GroupBy agg",
     lambda p,f: spark.read.format(f).load(p).groupBy("category").agg(F.sum("revenue")).collect()),
]

print(f"{'Query':<30} {'Avro':>10} {'Parquet':>10} {'Parquet faster':>15}")
print("-" * 68)

for label, fn in benchmarks:
    ta, tp = [], []
    for _ in range(3):
        t0=time.time(); fn(AVRO_BENCH,"avro");    ta.append(time.time()-t0)
        t0=time.time(); fn(PQ_BENCH,"parquet");   tp.append(time.time()-t0)
    t_a, t_p = sorted(ta)[1], sorted(tp)[1]
    faster = f"{t_a/t_p:.1f}x" if t_p < t_a else f"{t_p/t_a:.1f}x slower"
    print(f"  {label:<28} {t_a:>8.3f}s  {t_p:>8.3f}s  {faster:>15}")


Query                                Avro    Parquet  Parquet faster
--------------------------------------------------------------------


  Full scan + sum                 0.390s     0.412s      1.1x slower
  Filter + count                  0.345s     0.368s      1.1x slower
  Select 2 cols                   0.309s     0.328s      1.1x slower
  GroupBy agg                     0.329s     0.382s      1.2x slower


## Step 3 — Column Pruning: The Key Difference



In [4]:

# Avro: must read all bytes regardless of column selection
# Parquet: reads only selected column byte ranges

print("Column pruning demonstration (SELECT 1 of 7 columns):")
print()

t_avro_full, t_avro_pruned, t_pq_pruned = [], [], []
for _ in range(3):
    t0=time.time(); spark.read.format("avro").load(AVRO_BENCH).agg(F.sum("revenue")).collect(); t_avro_full.append(time.time()-t0)
    t0=time.time(); spark.read.format("avro").load(AVRO_BENCH).select("revenue").agg(F.sum("revenue")).collect(); t_avro_pruned.append(time.time()-t0)
    t0=time.time(); spark.read.parquet(PQ_BENCH).select("revenue").agg(F.sum("revenue")).collect(); t_pq_pruned.append(time.time()-t0)

t_af = sorted(t_avro_full)[1]
t_ap = sorted(t_avro_pruned)[1]
t_pp = sorted(t_pq_pruned)[1]

print(f"  Avro SELECT *         : {t_af:.3f}s  (reads all bytes)")
print(f"  Avro SELECT revenue   : {t_ap:.3f}s  (still reads all bytes — no pruning!)")
print(f"  Parquet SELECT revenue: {t_pp:.3f}s  (reads only revenue column bytes)")
print()
print(f"Parquet column pruning speedup: {t_ap/t_pp:.1f}x")
print()
print("This is the #1 reason to use Parquet for analytics:")
print("  Parquet reads ~1/7 of the data for a 1-column query")
print("  Avro always reads 100% of the data regardless of columns selected")


Column pruning demonstration (SELECT 1 of 7 columns):

  Avro SELECT *         : 0.264s  (reads all bytes)
  Avro SELECT revenue   : 0.262s  (still reads all bytes — no pruning!)
  Parquet SELECT revenue: 0.299s  (reads only revenue column bytes)

Parquet column pruning speedup: 0.9x

This is the #1 reason to use Parquet for analytics:
  Parquet reads ~1/7 of the data for a 1-column query
  Avro always reads 100% of the data regardless of columns selected


## Step 4 — When to Use Each



In [5]:

print("""
Decision guide:

Use AVRO when:
  ✅ Kafka / Pulsar message serialization
  ✅ Row-level access patterns (read single records)
  ✅ Streaming ingestion (fast row-by-row writes)
  ✅ Schema evolution needed across many producers/consumers
  ✅ Language-agnostic serialization (Java, Python, Go, C++)
  ✅ Landing zone for raw event data

Use PARQUET when:
  ✅ Analytical queries (SELECT few cols, aggregate)
  ✅ Data lake storage (long-term)
  ✅ Query engines (Spark SQL, Trino, Athena, BigQuery)
  ✅ Column pruning needed (wide tables with few queried cols)
  ✅ Predicate pushdown needed (filter on statistics)
  ✅ Compressed storage (Parquet typically 3-5x smaller than Avro)

Best practice: Kafka → Avro → Spark → Parquet
  1. Producers write Avro to Kafka (schema-safe, fast)
  2. Spark reads Avro from Kafka (or landing zone)
  3. Spark writes Parquet (analytics zone, partitioned)
  4. Analysts query Parquet (fast, cheap)
""")



Decision guide:

Use AVRO when:
  ✅ Kafka / Pulsar message serialization
  ✅ Row-level access patterns (read single records)
  ✅ Streaming ingestion (fast row-by-row writes)
  ✅ Schema evolution needed across many producers/consumers
  ✅ Language-agnostic serialization (Java, Python, Go, C++)
  ✅ Landing zone for raw event data

Use PARQUET when:
  ✅ Analytical queries (SELECT few cols, aggregate)
  ✅ Data lake storage (long-term)
  ✅ Query engines (Spark SQL, Trino, Athena, BigQuery)
  ✅ Column pruning needed (wide tables with few queried cols)
  ✅ Predicate pushdown needed (filter on statistics)
  ✅ Compressed storage (Parquet typically 3-5x smaller than Avro)

Best practice: Kafka → Avro → Spark → Parquet
  1. Producers write Avro to Kafka (schema-safe, fast)
  2. Spark reads Avro from Kafka (or landing zone)
  3. Spark writes Parquet (analytics zone, partitioned)
  4. Analysts query Parquet (fast, cheap)



## Summary



In [6]:

print("""
Format comparison:
  Feature            Avro           Parquet
  ─────────────────────────────────────────
  Layout             Row-based      Columnar
  Column pruning     ❌ No          ✅ Yes
  Predicate pushdown ❌ No stats    ✅ Min/max stats
  Schema in file     ✅ JSON header ✅ Binary footer
  Schema evolution   ✅ First-class ⚠️  Limited
  Streaming writes   ✅ Fast        ✅ OK
  Analytics reads    ❌ Slow        ✅ Fast
  Compression ratio  Medium         High (columnar)
  Kafka integration  ✅ Native      ❌ Unusual
""")



Format comparison:
  Feature            Avro           Parquet
  ─────────────────────────────────────────
  Layout             Row-based      Columnar
  Column pruning     ❌ No          ✅ Yes
  Predicate pushdown ❌ No stats    ✅ Min/max stats
  Schema in file     ✅ JSON header ✅ Binary footer
  Schema evolution   ✅ First-class ⚠️  Limited
  Streaming writes   ✅ Fast        ✅ OK
  Analytics reads    ❌ Slow        ✅ Fast
  Compression ratio  Medium         High (columnar)
  Kafka integration  ✅ Native      ❌ Unusual

